In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import pickle

from tqdm import tqdm

# --- 1. Setup Kaggle Paths ---
# In Kaggle, datasets are usually in /kaggle/input
# Adjust 'dataset-name' to match your actual Kaggle dataset name
INPUT_PATH = '/kaggle/input/datasets/danofer/sarcasm/train-balanced-sarcasm.csv' 
OUTPUT_PATH = '/kaggle/working/'

# Ensure output path exists
if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)

# --- 2. Load Data ---
print("Loading data...")
df = pd.read_csv(INPUT_PATH)
print(f"Dataset loaded: {df.shape[0]} rows")

# --- 3. Data Cleaning ---
print("Cleaning data...")
# Remove missing values
df.dropna(subset=['comment', 'parent_comment'], inplace=True)
print(f"Rows after dropping NaNs: {df.shape[0]}")

# Text cleaning optimized for sarcasm detection
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '[URL]', text, flags=re.MULTILINE)
    # Remove extra whitespace but preserve punctuation and emojis
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Applying cleaning...")
tqdm.pandas()
df['comment'] = df['comment'].progress_apply(clean_text)
df['parent_comment'] = df['parent_comment'].progress_apply(clean_text)

# --- 4. Context Merging (Phase 1 Core) ---
# For Phase 2 (RAG/Model), having context is crucial.
# We create a combined field: "Context: [parent] Reply: [comment]"
print("Merging context...")
df['combined_text'] = "Context: " + df['parent_comment'] + " Reply: " + df['comment']

# --- 5. Exploratory Data Analysis (EDA) ---
print("Running EDA...")

# A. Class Balance
plt.figure(figsize=(6, 4))
sns.countplot(x='label', data=df, palette='viridis')
plt.title('Class Distribution (0: Non-Sarcastic, 1: Sarcastic)')
plt.savefig(os.path.join(OUTPUT_PATH, 'class_distribution.png'))
plt.show()

# B. Comment Length Analysis
df['comment_len'] = df['comment'].apply(lambda x: len(x.split()))
plt.figure(figsize=(10, 6))
sns.histplot(df[df['label'] == 1]['comment_len'], label='Sarcastic', color='red', kde=True, bins=50)
sns.histplot(df[df['label'] == 0]['comment_len'], label='Non-Sarcastic', color='blue', kde=True, bins=50)
plt.title('Comment Length Distribution')
plt.xlim(0, 100) # Most comments are short
plt.legend()
plt.savefig(os.path.join(OUTPUT_PATH, 'length_distribution.png'))
plt.show()

# C. Top Subreddits for Sarcasm
top_sarcasm_subreddits = df[df['label'] == 1]['subreddit'].value_counts().head(10)
plt.figure(figsize=(12, 6))
top_sarcasm_subreddits.plot(kind='bar', color='orange')
plt.title('Top 10 Subreddits with Highest Sarcasm Counts')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'top_subreddits.png'))
plt.show()

# --- 6. Save Results for Phase 2 ---
# We save a subset if the data is too large, or the full thing if memory allows.
# For Kaggle, saving to pickle is efficient for Phase 2.
print("Saving preprocessed data...")
# Keeping only necessary columns for modeling to save space
model_df = df[['label', 'comment', 'parent_comment', 'combined_text', 'subreddit']]

# Save as pickle for faster loading in the next phase
save_path = os.path.join(OUTPUT_PATH, 'preprocessed_data.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(model_df, f)

print(f"Phase 1 Complete. Results saved to {save_path}")

